# Notebook 02 — 졸음/부주의 감지 모듈

## 목표
- PERCLOS 알고리즘 구현 (시간 기반 졸음 판단)
- YOLOv8로 휴대폰/담배 감지
- 멀티 신호 통합 → 경고 레벨 1/2/3 결정
- LangGraph Agent에 넘길 표준 State 설계

## 스킬업 포인트
- PERCLOS: 자율주행 업계 표준 졸음 지표
- 멀티 신호 퓨전: 단일 지표보다 정확도 향상
- State 설계: LangGraph 연동의 핵심

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import cv2
import numpy as np
import mediapipe as mp
from scipy.spatial import distance as dist
from collections import deque
import time
import matplotlib.pyplot as plt

print('패키지 로드 완료')

## 1. Notebook 01 함수 재사용

실무에서는 공통 모듈을 import해서 씁니다. 여기서는 tools/ 폴더에 분리해뒀어요.

In [ ]:
import sys
sys.path.append('..')

# 눈/입 랜드마크 인덱스
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH     = [61, 291, 13, 14, 17, 0, 402, 178]

# 임계값
EAR_THRESH    = 0.25
MAR_THRESH    = 0.60
YAW_THRESH    = 30.0
PITCH_THRESH  = 20.0

def calculate_EAR(pts):
    A = dist.euclidean(pts[1], pts[5])
    B = dist.euclidean(pts[2], pts[4])
    C = dist.euclidean(pts[0], pts[3])
    return (A + B) / (2.0 * C)

def calculate_MAR(pts):
    A = dist.euclidean(pts[1], pts[7])
    B = dist.euclidean(pts[2], pts[6])
    C = dist.euclidean(pts[3], pts[5])
    D = dist.euclidean(pts[0], pts[4])
    return (A + B + C) / (2.0 * D)

def get_head_pose(landmarks, shape):
    h, w = shape[:2]
    model_pts = np.array([
        [0.0, 0.0, 0.0], [0.0, -330.0, -65.0],
        [-225.0, 170.0, -135.0], [225.0, 170.0, -135.0],
        [-150.0, -150.0, -125.0], [150.0, -150.0, -125.0]
    ], dtype=np.float64)
    key_idx = [1, 152, 33, 263, 61, 291]
    pts_2d = np.array([[landmarks[i].x*w, landmarks[i].y*h] for i in key_idx], dtype=np.float64)
    fl = w
    cam = np.array([[fl,0,w/2],[0,fl,h/2],[0,0,1]], dtype=np.float64)
    ok, rvec, _ = cv2.solvePnP(model_pts, pts_2d, cam, np.zeros((4,1)))
    if not ok: return None, None, None
    rmat, _ = cv2.Rodrigues(rvec)
    angles, *_ = cv2.RQDecomp3x3(rmat)
    return angles[0]*360, angles[1]*360, angles[2]*360

print('공통 함수 로드 완료')

## 2. PERCLOS — 자율주행 업계 표준 졸음 지표

**PERCLOS** = PERcentage of eye CLOSure

- 단위 시간(보통 1분) 동안 눈이 80% 이상 감겨있는 시간의 비율
- PERCLOS > 15% → 졸음 운전 판정
- 단순 EAR보다 **오탐이 적고 신뢰도가 높음** (깜빡임 vs 졸음 구분)

In [ ]:
class PERCLOSCalculator:
    """
    PERCLOS 계산기
    - window_sec: 관찰 윈도우 (초)
    - fps: 카메라 FPS
    - threshold: 졸음 판정 PERCLOS 값
    """
    def __init__(self, window_sec=60, fps=30, threshold=0.15):
        self.window_size = window_sec * fps
        self.threshold   = threshold
        self.ear_buffer  = deque(maxlen=self.window_size)

    def update(self, ear):
        """새 EAR 값 추가 후 현재 PERCLOS 반환"""
        self.ear_buffer.append(ear if ear is not None else 1.0)
        closed_frames = sum(1 for e in self.ear_buffer if e < EAR_THRESH)
        perclos = closed_frames / len(self.ear_buffer)
        return perclos

    @property
    def is_drowsy(self):
        return self.update.__wrapped__ if hasattr(self.update, '__wrapped__') else False

    def get_perclos(self):
        if not self.ear_buffer:
            return 0.0
        return sum(1 for e in self.ear_buffer if e < EAR_THRESH) / len(self.ear_buffer)


# PERCLOS 시뮬레이션
np.random.seed(42)
fps = 30

# 정상(30초) → 졸음(20초) → 정상(10초)
ear_sim = np.concatenate([
    np.random.normal(0.32, 0.02, 30*fps),
    np.random.normal(0.16, 0.04, 20*fps),
    np.random.normal(0.31, 0.02, 10*fps),
])

calc = PERCLOSCalculator(window_sec=30, fps=fps, threshold=0.15)
perclos_values = [calc.update(e) for e in ear_sim]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle('PERCLOS 알고리즘 시뮬레이션', fontsize=13, fontweight='bold')

frames = np.arange(len(ear_sim))
ax1.plot(frames/fps, ear_sim, 'b-', linewidth=0.8, alpha=0.7)
ax1.axhline(EAR_THRESH, color='r', linestyle='--', label='EAR 임계값')
ax1.set_ylabel('EAR')
ax1.legend()
ax1.set_title('원시 EAR 신호')

ax2.plot(frames/fps, perclos_values, 'orange', linewidth=2)
ax2.axhline(0.15, color='r', linestyle='--', label='PERCLOS 임계값 (15%)')
ax2.fill_between(frames/fps, perclos_values, 0.15,
                 where=np.array(perclos_values)>0.15,
                 alpha=0.4, color='red', label='졸음 구간')
ax2.set_ylabel('PERCLOS')
ax2.set_xlabel('시간 (초)')
ax2.legend()
ax2.set_title('PERCLOS (30초 윈도우)')

plt.tight_layout()
plt.savefig('../output/02_perclos.png', dpi=150, bbox_inches='tight')
plt.show()
print('PERCLOS 시뮬레이션 완료')

## 3. YOLOv8 — 휴대폰/담배 감지

Pretrained COCO 모델로 바로 사용 가능:
- cell phone (67번 클래스)
- cigarette은 COCO에 없어서 별도 파인튜닝 필요 (여기선 phone만)

**스킬업**: YOLOv8은 `ultralytics` 패키지로 3줄이면 추론 가능

In [ ]:
from ultralytics import YOLO
import urllib.request
from pathlib import Path

# 모델 다운로드 (최초 1회)
model_path = Path('../models/yolov8n.pt')
if not model_path.exists():
    print('YOLOv8n 다운로드 중... (6MB)')
    model_path.parent.mkdir(exist_ok=True)

yolo_model = YOLO('yolov8n.pt')  # 없으면 자동 다운로드
print(f'YOLOv8 로드 완료: {yolo_model.info()}')

# COCO 클래스 중 운전 위험 항목
DANGEROUS_CLASSES = {
    67: 'cell phone',
    73: 'book',       # 책 보는 행위
    76: 'scissors',   # 위험 물건
}

print('\n감지 대상 클래스:')
for cls_id, cls_name in DANGEROUS_CLASSES.items():
    print(f'  {cls_id}: {cls_name}')

In [ ]:
def detect_dangerous_objects(frame, model, conf_thresh=0.5):
    """
    프레임에서 위험 객체 감지
    Returns: list of {'class': str, 'confidence': float, 'bbox': [x1,y1,x2,y2]}
    """
    results = model(frame, verbose=False)[0]
    detections = []

    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        if cls_id in DANGEROUS_CLASSES and conf >= conf_thresh:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            detections.append({
                'class':      DANGEROUS_CLASSES[cls_id],
                'confidence': round(conf, 3),
                'bbox':       [x1, y1, x2, y2]
            })
    return detections


def draw_detections(frame, detections):
    """감지된 객체를 프레임에 시각화"""
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        label = f"{det['class']} {det['confidence']:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(frame, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return frame

print('YOLO 감지 함수 정의 완료')

## 4. 멀티 신호 퓨전 — 경고 레벨 결정 로직

단일 지표보다 여러 지표를 종합하면 정확도가 올라갑니다.

| 조건 | 레벨 | 의미 |
|------|------|---------|
| 모두 정상 | 0 | 정상 |
| 1개 이상 이상 | 1 | 주의 |
| 2개 이상 이상 OR PERCLOS>15% | 2 | 경고 |
| 3개 이상 이상 OR 위험물체 감지 | 3 | 위험 |

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List


@dataclass
class DriverState:
    """
    운전자 상태 — LangGraph Agent 간 공유되는 핵심 State
    Notebook 03에서 TypedDict로 변환 예정
    """
    # 원시 지표
    ear:     Optional[float] = None   # 눈 감김 (0~1, 낮을수록 감김)
    mar:     Optional[float] = None   # 입 벌림 (0~1, 높을수록 하품)
    pitch:   Optional[float] = None   # 고개 상하 각도
    yaw:     Optional[float] = None   # 고개 좌우 각도
    perclos: Optional[float] = None   # 시간 기반 졸음 비율
    objects: List[dict]      = field(default_factory=list)  # YOLO 감지 결과

    # 판단 결과
    face_detected:   bool = False
    is_drowsy:       bool = False
    is_distracted:   bool = False
    is_yawning:      bool = False
    has_danger_obj:  bool = False
    alert_level:     int  = 0         # 0=정상, 1=주의, 2=경고, 3=위험
    alert_reason:    str  = ''
    llm_message:     str  = ''
    timestamp:       float = field(default_factory=time.time)


class SignalFusion:
    """멀티 신호 퓨전 — 경고 레벨 결정"""

    def __init__(self):
        self.perclos_calc = PERCLOSCalculator(window_sec=30, fps=30)

    def fuse(self, raw: dict) -> DriverState:
        state = DriverState(
            ear=raw.get('ear'), mar=raw.get('mar'),
            pitch=raw.get('pitch'), yaw=raw.get('yaw'),
            objects=raw.get('objects', []),
            face_detected=raw.get('detected', False)
        )

        if not state.face_detected:
            state.alert_level = 1
            state.alert_reason = '얼굴 미감지'
            return state

        # PERCLOS 업데이트
        state.perclos = self.perclos_calc.update(state.ear)

        # 개별 판단
        state.is_drowsy    = (state.ear < EAR_THRESH) or (state.perclos > 0.15)
        state.is_yawning   = (state.mar > MAR_THRESH)
        state.is_distracted = (state.yaw is not None and abs(state.yaw) > YAW_THRESH) or \
                              (state.pitch is not None and state.pitch > PITCH_THRESH)
        state.has_danger_obj = len(state.objects) > 0

        # 경고 레벨 결정
        risk_count = sum([
            state.is_drowsy, state.is_yawning,
            state.is_distracted, state.has_danger_obj
        ])

        if state.has_danger_obj:
            state.alert_level  = 3
            state.alert_reason = f"위험 물체 감지: {state.objects[0]['class']}"
        elif state.perclos > 0.15 or risk_count >= 3:
            state.alert_level  = 3
            state.alert_reason = '심각한 졸음 운전'
        elif risk_count >= 2:
            state.alert_level  = 2
            state.alert_reason = '복합 위험 신호'
        elif risk_count == 1:
            state.alert_level  = 1
            state.alert_reason = '주의 필요'
        else:
            state.alert_level  = 0
            state.alert_reason = '정상'

        return state


fusion = SignalFusion()
print('SignalFusion 정의 완료')
print('\n테스트:')
test_raw = {'detected': True, 'ear': 0.18, 'mar': 0.65,
            'pitch': 5.0, 'yaw': 8.0, 'objects': []}
result = fusion.fuse(test_raw)
print(f'  EAR={test_raw["ear"]}, MAR={test_raw["mar"]}')
print(f'  → 졸음:{result.is_drowsy}, 하품:{result.is_yawning}')
print(f'  → 경고 레벨: {result.alert_level} ({result.alert_reason})')

## 5. 경고 레벨별 시각화

In [ ]:
def get_alert_color(level):
    return {0: (0,200,0), 1: (0,165,255), 2: (0,100,255), 3: (0,0,255)}[level]

def get_alert_text(level):
    return {0:'NORMAL', 1:'CAUTION', 2:'WARNING', 3:'DANGER!'}[level]

def render_dashboard(state: DriverState, frame_size=(500,300)):
    """상태 대시보드 이미지 생성 (시각화용)"""
    img = np.zeros((*frame_size, 3), dtype=np.uint8)
    img[:] = (30, 30, 30)

    color = get_alert_color(state.alert_level)
    text  = get_alert_text(state.alert_level)

    # 경고 레벨 배경
    cv2.rectangle(img, (0,0), (500,60), color, -1)
    cv2.putText(img, text, (15, 44),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 3)
    cv2.putText(img, state.alert_reason, (15, 85),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

    # 지표 표시
    metrics = [
        (f"EAR    : {state.ear:.3f}" if state.ear else 'EAR: N/A',
         (0,0,255) if state.is_drowsy else (0,200,0)),
        (f"MAR    : {state.mar:.3f}" if state.mar else 'MAR: N/A',
         (0,0,255) if state.is_yawning else (0,200,0)),
        (f"Yaw    : {state.yaw:.1f}°" if state.yaw else 'Yaw: N/A',
         (0,0,255) if state.is_distracted else (0,200,0)),
        (f"PERCLOS: {state.perclos*100:.1f}%" if state.perclos else 'PERCLOS: N/A',
         (0,0,255) if (state.perclos or 0)>0.15 else (0,200,0)),
    ]
    for i, (txt, col) in enumerate(metrics):
        cv2.putText(img, txt, (15, 130+i*35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, col, 2)

    if state.has_danger_obj:
        cv2.putText(img, f"OBJECT : {state.objects[0]['class']}",
                    (15, 270), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,255), 2)
    return img


# 4가지 레벨 시각화
scenarios = [
    {'detected':True, 'ear':0.32, 'mar':0.2, 'pitch':3,'yaw':5,  'objects':[]},
    {'detected':True, 'ear':0.22, 'mar':0.2, 'pitch':3,'yaw':5,  'objects':[]},
    {'detected':True, 'ear':0.20, 'mar':0.65,'pitch':3,'yaw':35, 'objects':[]},
    {'detected':True, 'ear':0.15, 'mar':0.7, 'pitch':3,'yaw':40,
     'objects':[{'class':'cell phone','confidence':0.92,'bbox':[100,100,200,200]}]},
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('경고 레벨별 대시보드', fontsize=13, fontweight='bold')

for ax, raw in zip(axes, scenarios):
    st = fusion.fuse(raw)
    img = render_dashboard(st)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'레벨 {st.alert_level}: {get_alert_text(st.alert_level)}')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../output/02_alert_levels.png', dpi=150, bbox_inches='tight')
plt.show()
print('대시보드 시각화 완료')

## 정리

이번 노트북에서 만든 것:
- `PERCLOSCalculator` — 시간 기반 졸음 판단기
- `SignalFusion` — 멀티 신호 통합 클래스
- `DriverState` — 에이전트 간 공유 상태 객체
- `render_dashboard` — 실시간 대시보드

**다음 Notebook 03**: 이 컴포넌트들을 LangGraph 에이전트로 래핑